In [ ]:
## Cell 0 — 환경
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install -U transformers accelerate scikit-learn

from google.colab import drive, userdata
drive.mount('/content/drive')
import os, json, time, hashlib
import numpy as np, pandas as pd, torch, transformers
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
print(torch.__version__, transformers.__version__)

name, memory.total [MiB]
NVIDIA A100-SXM4-80GB, 81920 MiB
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
2.11.0+cu128 5.17.0


In [ ]:
## Cell 1 — 경로 확인 (폴더명 p8s_v2 / hidden_v2 중 뭔지)
P9 = "/content/drive/MyDrive/Grad_share/data_share/P9/"
for cand in ["p8s_v2/", "hidden_v2/"]:
    if os.path.isdir(P9+cand):
        P = P9+cand; print("찾음:", P)
        for r,_,f in os.walk(P):
            for x in f: print("  ", os.path.join(r,x).replace(P,""))
        break

찾음: /content/drive/MyDrive/Grad_share/data_share/P9/hidden_v2/
   p8s_v2_mlp_out.npy
   p8s_v2_h_out.npy
   p8s_v2_post_attn_ln_weight.npy
   p8s_v2_attn_out.npy
   p8s_v2_meta.json
   k_flip_audit.csv
   k_relabel_v2.parquet
   p8s_behavior_v2.csv
   p8s_behavior_v2.gsheet
   p8s_v2_pks.npy
   p8s_v2_acs.npy
   p8s_v2_mlpnorm.npy
   steer_baseline.parquet
   steer_w.npy
   steer_all.parquet
   steer_cplus.parquet


In [ ]:
## Cell 2 — 데이터 로드
HD = P + ("hidden/" if os.path.isdir(P+"hidden") else "")
LB = P + ("labels/" if os.path.isdir(P+"labels") else "")

m = pd.read_csv(LB + "p8s_behavior_v2.csv")
H = {k: np.load(HD + f"p8s_v2_{k}.npy", mmap_mode="r") for k in ("h_out","mlp_out")}
meta = json.load(open(HD + "p8s_v2_meta.json"))

assert len(H["h_out"]) == len(m) == 1224
print(H["h_out"].shape, H["h_out"].dtype)
print(m.cell_k.value_counts().to_dict())
print(meta["layer_index"], "|", meta["prompt_version"])

(1224, 32, 4096) float16
{'K+C+': 346, 'K+C-': 346, 'K−C+': 266, 'K−C-': 266}
0-based; index i == model.model.layers[i] | p8s-v2


In [ ]:
## Cell 3 — 모델 로드 (lm_head + final norm만 쓰지만 전체 필요)
from transformers import AutoModelForCausalLM
MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, device_map="auto",
    attn_implementation="sdpa", token=os.environ["HF_TOKEN"]).eval()
final_norm, lm_head = model.model.norm, model.lm_head
print(model.dtype, lm_head.weight.shape, len(model.model.layers))

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

torch.bfloat16 torch.Size([128256, 4096]) 32


In [ ]:
## Cell F2 — PKS / ACS / mlp_norm 계산
N, NL, D = H["h_out"].shape
BS = 16
PKS  = np.zeros((N, NL), np.float32)
ACS  = np.zeros((N, NL), np.float32)
MLPN = np.zeros((N, NL), np.float32)

def jsd(lp, lq):
    p, q = torch.softmax(lp, -1), torch.softmax(lq, -1)
    mlog = ((p + q) / 2).clamp_min(1e-12).log()
    kl = lambda x: (x * (x.clamp_min(1e-12).log() - mlog)).sum(-1)
    return (0.5 * (kl(p) + kl(q))).clamp_min(0) / np.log(2)

t0 = time.time()
with torch.inference_mode():
    for s in range(0, N, BS):
        e = min(s + BS, N)
        ho = torch.from_numpy(np.asarray(H["h_out"][s:e])).to(model.device, torch.float32)
        mo = torch.from_numpy(np.asarray(H["mlp_out"][s:e])).to(model.device, torch.float32)
        hm = ho - mo
        MLPN[s:e] = (mo.norm(dim=-1) / hm.norm(dim=-1).clamp_min(1e-6)).cpu().numpy()

        prev = None
        for l in range(NL):
            lo  = lm_head(final_norm(ho[:, l].to(model.dtype))).float()
            lmd = lm_head(final_norm(hm[:, l].to(model.dtype))).float()
            PKS[s:e, l] = jsd(lmd, lo).cpu().numpy()
            if prev is not None:
                ACS[s:e, l] = jsd(prev, lmd).cpu().numpy()
            prev = lo
        del ho, mo, hm, lo, lmd, prev
        if s % (BS*20) == 0:
            print(f"  {s}/{N} {time.time()-t0:.0f}s {torch.cuda.memory_allocated()/1e9:.1f}GB", flush=True)

np.save(P + "p8s_v2_pks.npy", PKS)
np.save(P + "p8s_v2_acs.npy", ACS)
np.save(P + "p8s_v2_mlpnorm.npy", MLPN)
print(f"done {time.time()-t0:.0f}s")
print("PKS  layer 0/7/15/23/31:", np.round(PKS.mean(0)[[0,7,15,23,31]], 4))
print("ACS  layer 1/7/15/23/31:", np.round(ACS.mean(0)[[1,7,15,23,31]], 4))
print("MLPN layer 0/7/15/23/31:", np.round(MLPN.mean(0)[[0,7,15,23,31]], 4))

  0/1224 0s 16.1GB
  320/1224 2s 16.1GB
  640/1224 4s 16.1GB


In [ ]:
## Cell F2b — 전체 곡선
for name, F in [("PKS", PKS), ("ACS", ACS), ("MLPN", MLPN)]:
    print(name, np.round(F.mean(0), 3))

In [ ]:
## Cell F3 — 층별 K± AUROC (C 층화)
from sklearn.metrics import roc_auc_score
NL = PKS.shape[1]

rows = []
for c in ["C+", "C-"]:
    sel = (m.ctx_label == c).values
    y   = (m.kside_k[sel] == "K+").values.astype(int)
    for name, F in [("PKS", PKS), ("ACS", ACS), ("MLPN", MLPN)]:
        for l in range(1, NL):
            rows.append(dict(C=c, feat=name, layer=l,
                             auroc=round(roc_auc_score(y, F[sel, l]), 3)))
R = pd.DataFrame(rows)
print(R.pivot_table(index="layer", columns=["C","feat"], values="auroc").to_string())
print("\n최대:")
print(R.loc[R.groupby(["C","feat"]).auroc.idxmax()].to_string(index=False))

In [ ]:
## Cell F4 — 통제군: 노름비가 아니라 분모/분자 하나만으로 되는 건 아닌지
NRM_MID = np.zeros((N, NL), np.float32)
NRM_MLP = np.zeros((N, NL), np.float32)
for s in range(0, N, 64):
    e = min(s+64, N)
    ho = np.asarray(H["h_out"][s:e]).astype(np.float32)
    mo = np.asarray(H["mlp_out"][s:e]).astype(np.float32)
    NRM_MID[s:e] = np.linalg.norm(ho - mo, axis=-1)
    NRM_MLP[s:e] = np.linalg.norm(mo, axis=-1)

sel = (m.ctx_label == "C-").values
y   = (m.kside_k[sel] == "K+").values.astype(int)
for name, F in [("MLPN(비)", MLPN), ("‖mlp_out‖", NRM_MLP), ("‖h_mid‖", NRM_MID)]:
    a = [roc_auc_score(y, F[sel, l]) for l in range(1, NL)]
    print(f"{name:11s} L22-27 평균={np.mean(a[21:27]):.3f}  최대={max(a):.3f} (L{int(np.argmax(a))+1})")

In [ ]:
## Cell F5 — 사전 지정 구간 + 부트스트랩 CI
def boot(y, s, q, n=2000, seed=0):
    rng = np.random.RandomState(seed); uq = np.unique(q); out=[]
    for _ in range(n):
        idx = np.concatenate([np.where(q==u)[0] for u in rng.choice(uq, len(uq), True)])
        if len(np.unique(y[idx])) > 1: out.append(roc_auc_score(y[idx], s[idx]))
    return np.percentile(out, [2.5, 97.5])

for c in ["C+", "C-"]:
    sel = (m.ctx_label == c).values
    y, q = (m.kside_k[sel]=="K+").values.astype(int), m.question[sel].values
    for name, F, band in [("MLPN", MLPN, range(22,28)), ("PKS", PKS, range(17,22))]:
        sc = F[sel][:, list(band)].mean(1)          # 구간 평균 = 사전 지정
        lo, hi = boot(y, sc, q)
        print(f"{c} {name} L{band[0]}-{band[-1]} 평균: AUROC={roc_auc_score(y,sc):.3f} [{lo:.3f}, {hi:.3f}]")

In [ ]:
## Cell F6 — 교란 검사
sel = (m.ctx_label=="C-").values
y   = (m.kside_k[sel]=="K+").values.astype(int)
q   = m.question[sel].values
S   = NRM_MLP[sel][:, 22:28].mean(1)

# ① massive activation 지배 여부
mo = np.asarray(H["mlp_out"][:, 22:28]).astype(np.float32)
share = (np.sort(mo**2, -1)[..., -10:].sum(-1) / (mo**2).sum(-1)).mean()
print(f"상위 10개 차원이 차지하는 에너지 비중: {share:.3f}")

# 상위 10차원 제거 후에도 남는가
topd = np.argsort((mo**2).mean((0,1)))[-10:]
mask = np.ones(4096, bool); mask[topd] = False
S_masked = np.linalg.norm(mo[sel][..., mask], axis=-1).mean(1)
print(f"상위10 제거 후 AUROC: {roc_auc_score(y, S_masked):.3f}  (원본 {roc_auc_score(y,S):.3f})")

# ② 컨텍스트 길이 교란
L = m.ctx_text[sel].str.len().values
print(f"\n‖mlp_out‖ vs ctx 길이 상관: {np.corrcoef(S, L)[0,1]:.3f}")
print(f"ctx 길이만으로 AUROC: {roc_auc_score(y, L):.3f}")

# ③ prop별로도 유지되는가 (속성 편향이 아닌지)
pr = m.prop[sel].values
for p in pd.Series(pr).value_counts().head(6).index:
    i = pr == p
    if len(np.unique(y[i])) > 1:
        print(f"  {p:15s} n={i.sum():3d}  AUROC={roc_auc_score(y[i], S[i]):.3f}")

In [ ]:
## Cell F7 — 길이 통제
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

X = StandardScaler().fit_transform(np.c_[S, L])
p_both = LogisticRegression(max_iter=1000).fit(X, y).predict_proba(X)[:,1]
p_len  = LogisticRegression(max_iter=1000).fit(X[:,[1]], y).predict_proba(X[:,[1]])[:,1]
print(f"길이만        {roc_auc_score(y, p_len):.3f}")
print(f"‖mlp_out‖만   {roc_auc_score(y, S):.3f}")
print(f"둘 다         {roc_auc_score(y, p_both):.3f}")

# 길이 5분위 안에서 각각
qs_ = pd.qcut(L, 5, labels=False)
print("\n길이 5분위별 ‖mlp_out‖ AUROC:")
for b in range(5):
    i = qs_ == b
    if len(np.unique(y[i])) > 1:
        print(f"  Q{b+1} n={i.sum():3d} 평균길이={L[i].mean():5.0f}  AUROC={roc_auc_score(y[i], S[i]):.3f}")

In [ ]:
## Cell F8 — 앞층에서도 되나
for band, name in [(range(8,14),"L8-13"), (range(14,19),"L14-18"), (range(22,28),"L22-27")]:
    sc = NRM_MLP[sel][:, list(band)].mean(1)
    print(f"{name:8s} AUROC={roc_auc_score(y, sc):.3f}")

In [ ]:
## Cell F9 — 부호 전환점 찾기
a = [roc_auc_score(y, NRM_MLP[sel, l]) for l in range(32)]
print(" ".join(f"{l}:{v:.2f}" for l, v in enumerate(a)))
print("\n0.5 교차:", [l for l in range(1,32) if (a[l-1]-0.5)*(a[l]-0.5) < 0])

In [ ]:
## Cell F10 — L13 이하만으로 만든 feature들
def auc(v): return roc_auc_score(y, v)

f_early = NRM_MLP[sel][:, 8:14].mean(1)
f_veryE = NRM_MLP[sel][:, 2:8].mean(1)
print(f"L8-13          {auc(f_early):.3f}  (뒤집으면 {1-auc(f_early):.3f})")
print(f"L2-7           {auc(f_veryE):.3f}  (뒤집으면 {1-auc(f_veryE):.3f})")
print(f"L8-13 - L2-7   {auc(f_early - f_veryE):.3f}")
print(f"L8-13 / L2-7   {auc(f_early / np.clip(f_veryE,1e-6,None)):.3f}")

# 층별 전부 던져서 상한 확인 (OOF, question 단위)
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegressionCV
for cut, nm in [(14,"≤L13"), (19,"≤L18"), (32,"전체")]:
    X = StandardScaler().fit_transform(NRM_MLP[sel][:, :cut])
    oof = np.zeros(len(y))
    for tr, te in GroupKFold(5).split(X, y, groups=q):
        oof[te] = LogisticRegressionCV(max_iter=2000).fit(X[tr], y[tr]).predict_proba(X[te])[:,1]
    print(f"OOF logistic {nm:6s} {roc_auc_score(y, oof):.3f}")

In [ ]:
## Cell F11 — leakage 수정 + 순열검정
import warnings; warnings.filterwarnings("ignore")
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression

def oof_auc(X, y, groups, seed=0):
    oof = np.zeros(len(y))
    for tr, te in GroupKFold(5).split(X, y, groups):
        pipe = make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=2000))
        pipe.fit(X[tr], y[tr])                       # 스케일러도 fold 안에서만
        oof[te] = pipe.predict_proba(X[te])[:, 1]
    return roc_auc_score(y, oof)

rng = np.random.RandomState(0)
uq = np.unique(q)
for cut, nm in [(14, "≤L13"), (19, "≤L18"), (32, "전체")]:
    X = NRM_MLP[sel][:, :cut]
    real = oof_auc(X, y, q)
    null = []
    for _ in range(100):
        perm = {u: v for u, v in zip(uq, rng.permutation(uq))}   # 질문 단위 셔플
        ymap = dict(zip(q, y))
        yp = np.array([ymap[perm[x]] for x in q])
        null.append(oof_auc(X, yp, q))
    p = (np.sum(np.array(null) >= real) + 1) / 101
    print(f"{nm:6s} OOF={real:.3f}  null={np.mean(null):.3f}±{np.std(null):.3f}  p={p:.3f}")

In [ ]:
## Cell F12 — cut 선택과 평가 분리
rng = np.random.RandomState(7)
uq2 = np.unique(q); rng.shuffle(uq2)
devq = set(uq2[:len(uq2)//2])
dev, tst = np.array([x in devq for x in q]), np.array([x not in devq for x in q])

best = max([(oof_auc(NRM_MLP[sel][dev][:, :c], y[dev], q[dev]), c)
            for c in (14, 19, 32)])
print(f"dev 최적 cut = ≤L{best[1]-1}  (dev AUROC {best[0]:.3f})")

X = NRM_MLP[sel][:, :best[1]]
pipe = make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=2000))
pipe.fit(X[dev], y[dev])
s_t = pipe.predict_proba(X[tst])[:, 1]
lo, hi = boot(y[tst], s_t, q[tst])
print(f"test AUROC = {roc_auc_score(y[tst], s_t):.3f} [{lo:.3f}, {hi:.3f}]  n={tst.sum()}")

In [ ]:
## Cell F13 — cut별 test 성능 (1-pass 가능 여부 최종 판정)
for cut, nm in [(14,"≤L13"), (19,"≤L18"), (32,"전체")]:
    X = NRM_MLP[sel][:, :cut]
    pipe = make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=2000))
    pipe.fit(X[dev], y[dev])
    s = pipe.predict_proba(X[tst])[:,1]
    lo, hi = boot(y[tst], s, q[tst])
    print(f"{nm:6s} test={roc_auc_score(y[tst], s):.3f} [{lo:.3f}, {hi:.3f}]")

In [ ]:
## Cell G1 — cross-C 평가 (게이트가 C를 몰라도 되는가)
import warnings; warnings.filterwarnings("ignore")
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score

CUT = 14                                     # L0-13
Xall = NRM_MLP[:, :CUT]
yall = (m.kside_k == "K+").values.astype(int)
qall = m.question.values
isC  = {c: (m.ctx_label == c).values for c in ["C+", "C-"]}

def fit_pred(tr_idx, te_idx):
    p = make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=2000))
    p.fit(Xall[tr_idx], yall[tr_idx])
    return p.predict_proba(Xall[te_idx])[:, 1]

# ① 같은 질문이 fold를 넘지 않게 하면서 train C / test C 조합
for trC in ["C-", "C+"]:
    for teC in ["C-", "C+"]:
        oof = np.zeros(len(yall)); msk = np.zeros(len(yall), bool)
        for tr, te in GroupKFold(5).split(Xall, yall, qall):
            tr2 = tr[isC[trC][tr]]; te2 = te[isC[teC][te]]
            oof[te2] = fit_pred(tr2, te2); msk[te2] = True
        print(f"train {trC} → test {teC}:  AUROC={roc_auc_score(yall[msk], oof[msk]):.3f}  n={msk.sum()}")

# ② C 구분 없이 통째로 (실제 배포 조건)
oof = np.zeros(len(yall))
for tr, te in GroupKFold(5).split(Xall, yall, qall):
    oof[te] = fit_pred(tr, te)
print(f"\nC 구분 없음 (배포 조건): AUROC={roc_auc_score(yall, oof):.3f}  n={len(yall)}")

# ③ 같은 질문의 C+/C− 점수가 안정적인가 (K 신호라면 안정적이어야)
piv = pd.DataFrame({"q": qall, "c": m.ctx_label, "s": oof}).pivot(index="q", columns="c", values="s")
print(f"같은 질문 C+/C− 게이트 점수 상관: {piv.corr().iloc[0,1]:.3f}")

In [ ]:
## Cell G2 — 질문 표면 정보만으로 얼마나 되나
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import cross_val_predict

qdf = m.drop_duplicates("question")
yq  = (qdf.kside_k == "K+").values.astype(int)
gq  = qdf.question.values

# ① 질문 길이 / 토큰수만
ql = np.c_[qdf.question.str.len(), qdf.question.str.split().str.len()]
oof = cross_val_predict(make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
                        ql, yq, cv=GroupKFold(5), groups=gq, method="predict_proba")[:,1]
print(f"질문 길이만          AUROC={roc_auc_score(yq, oof):.3f}")

# ② 질문 문자 n-gram (엔티티 이름 표면형)
tf = TfidfVectorizer(analyzer="char_wb", ngram_range=(3,5), max_features=5000)
oof = cross_val_predict(make_pipeline(tf, LogisticRegression(C=1, max_iter=2000)),
                        gq, yq, cv=GroupKFold(5), groups=gq, method="predict_proba")[:,1]
print(f"질문 텍스트만        AUROC={roc_auc_score(yq, oof):.3f}")

# ③ prop(속성) 더미만
pd_ = pd.get_dummies(qdf.prop).values.astype(float)
oof = cross_val_predict(LogisticRegression(max_iter=2000), pd_, yq, cv=GroupKFold(5),
                        groups=gq, method="predict_proba")[:,1]
print(f"속성(prop)만         AUROC={roc_auc_score(yq, oof):.3f}")

# ④ 질문 텍스트 + mlp_norm 결합이 mlp_norm 단독보다 나은가
Xq = NRM_MLP[m.drop_duplicates("question").index, :14]
oof_h = cross_val_predict(make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
                          Xq, yq, cv=GroupKFold(5), groups=gq, method="predict_proba")[:,1]
print(f"mlp_norm만 (질문단위) AUROC={roc_auc_score(yq, oof_h):.3f}")

In [ ]:
## Cell G3 — 증분 기여
from sklearn.model_selection import GroupKFold
import itertools

qdf = m.drop_duplicates("question").reset_index()
yq, gq = (qdf.kside_k=="K+").values.astype(int), qdf.question.values
Xh = NRM_MLP[qdf["index"].values, :14]
Xp = pd.get_dummies(qdf.prop).values.astype(float)
Xt = TfidfVectorizer(analyzer="char_wb", ngram_range=(3,5), max_features=5000)

def oof(X, y=yq, g=gq, sparse=False):
    p = make_pipeline(StandardScaler(with_mean=not sparse), LogisticRegression(C=1, max_iter=3000))
    o = np.zeros(len(y))
    for tr, te in GroupKFold(5).split(X, y, g):
        p.fit(X[tr], y[tr]); o[te] = p.predict_proba(X[te])[:,1]
    return roc_auc_score(y, o), o

for nm, X in [("prop", Xp), ("mlp", Xh), ("prop+mlp", np.c_[Xp, Xh])]:
    print(f"{nm:10s} {oof(X)[0]:.3f}")

# paired bootstrap: prop+mlp − prop
_, o_p  = oof(Xp)
_, o_pm = oof(np.c_[Xp, Xh])
rng = np.random.RandomState(0); uq = np.unique(gq); d = []
for _ in range(2000):
    idx = np.concatenate([np.where(gq==u)[0] for u in rng.choice(uq, len(uq), True)])
    if len(np.unique(yq[idx])) < 2: continue
    d.append(roc_auc_score(yq[idx], o_pm[idx]) - roc_auc_score(yq[idx], o_p[idx]))
print(f"\nΔ(prop+mlp − prop) = {np.mean(d):+.3f}  CI [{np.percentile(d,2.5):+.3f}, {np.percentile(d,97.5):+.3f}]")

# prop 내부에서만 (속성 고정 후에도 hidden이 되는가)
print("\n속성 고정 후 mlp_norm AUROC:")
for p_ in qdf.prop.value_counts().head(6).index:
    i = (qdf.prop == p_).values
    if len(np.unique(yq[i])) > 1 and i.sum() >= 30:
        a, _ = oof(Xh[i], yq[i], gq[i])
        print(f"  {p_:15s} n={i.sum():3d}  {a:.3f}")

In [ ]:
## Cell G4 — hidden 벡터 전체로 재시도 (속성 통제)
L = 13
Xv = np.asarray(H["h_out"][qdf["index"].values, L]).astype(np.float32)

print(f"전체        {oof(Xv)[0]:.3f}")
print("속성 고정:")
for p_ in qdf.prop.value_counts().head(6).index:
    i = (qdf.prop == p_).values
    if i.sum() >= 40 and len(np.unique(yq[i])) > 1:
        print(f"  {p_:15s} n={i.sum():3d}  {oof(Xv[i], yq[i], gq[i])[0]:.3f}")

In [ ]:
## Cell G5 — 결정적 검정 3개
Xv = np.asarray(H["h_out"][qdf["index"].values, 13]).astype(np.float32)
Xp = pd.get_dummies(qdf.prop).values.astype(float)

# ① prop 위 증분 (G3와 동일 프로토콜)
_, o_p  = oof(Xp)
_, o_pv = oof(np.c_[Xp, Xv])
print(f"prop {roc_auc_score(yq,o_p):.3f} → prop+hidden {roc_auc_score(yq,o_pv):.3f}")
rng = np.random.RandomState(0); uq = np.unique(gq); d=[]
for _ in range(2000):
    i = np.concatenate([np.where(gq==u)[0] for u in rng.choice(uq,len(uq),True)])
    if len(np.unique(yq[i]))<2: continue
    d.append(roc_auc_score(yq[i],o_pv[i]) - roc_auc_score(yq[i],o_p[i]))
print(f"Δ = {np.mean(d):+.3f}  CI [{np.percentile(d,2.5):+.3f}, {np.percentile(d,97.5):+.3f}]")

# ② 속성 내부 통합 AUROC + CI (prop별 OOF를 모아 prop별로 순위정규화 후 통합)
sc = np.full(len(yq), np.nan)
for p_ in qdf.prop.unique():
    i = (qdf.prop==p_).values
    if i.sum() >= 30 and len(np.unique(yq[i])) > 1:
        sc[i] = pd.Series(oof(Xv[i], yq[i], gq[i])[1]).rank(pct=True).values
ok = ~np.isnan(sc)
pooled = roc_auc_score(yq[ok], sc[ok])
b = [roc_auc_score(yq[ok][j], sc[ok][j]) for j in
     (rng.choice(ok.sum(), ok.sum(), True) for _ in range(2000))
     if len(np.unique(yq[ok][j]))>1]
print(f"\n속성내부 통합 AUROC = {pooled:.3f}  CI [{np.percentile(b,2.5):.3f}, {np.percentile(b,97.5):.3f}]  n={ok.sum()}")

# ③ 순열검정 (속성 내부 셔플)
null=[]
for _ in range(200):
    yp = yq.copy()
    for p_ in qdf.prop.unique():
        i = np.where((qdf.prop==p_).values)[0]
        yp[i] = rng.permutation(yq[i])
    s2 = np.full(len(yq), np.nan)
    for p_ in qdf.prop.unique():
        i = (qdf.prop==p_).values
        if i.sum()>=30 and len(np.unique(yp[i]))>1:
            s2[i] = pd.Series(oof(Xv[i], yp[i], gq[i])[1]).rank(pct=True).values
    o2 = ~np.isnan(s2)
    null.append(roc_auc_score(yp[o2], s2[o2]))
print(f"null = {np.mean(null):.3f}±{np.std(null):.3f}  p = {(np.sum(np.array(null)>=pooled)+1)/201:.4f}")

In [ ]:
## PKS도 prop 교란인지
Xk = PKS[qdf["index"].values, :14]
print(f"PKS 전체 {oof(Xk)[0]:.3f}")
_, o_pk = oof(np.c_[Xp, Xk])
d=[]
for _ in range(2000):
    i = np.concatenate([np.where(gq==u)[0] for u in rng.choice(uq,len(uq),True)])
    if len(np.unique(yq[i]))<2: continue
    d.append(roc_auc_score(yq[i],o_pk[i]) - roc_auc_score(yq[i],o_p[i]))
print(f"Δ(prop+PKS − prop) = {np.mean(d):+.3f} CI [{np.percentile(d,2.5):+.3f}, {np.percentile(d,97.5):+.3f}]")

In [ ]:
## Cell H1 — label-free 프록시가 되나
# ① 비지도: PCA 주성분이 K±와 맞는지
from sklearn.decomposition import PCA
Z = PCA(10).fit_transform(StandardScaler().fit_transform(Xv))
for j in range(5):
    a = roc_auc_score(yq, Z[:, j])
    print(f"PC{j+1}  AUROC={max(a, 1-a):.3f}")

# ② 프록시: closed-book 출력 자체의 불확실성
print(f"\nk_agree (샘플 일치도) AUROC={roc_auc_score(yq, qdf.k_agree):.3f}")
print(f"출력 길이             AUROC={roc_auc_score(yq, qdf.cb_greedy.str.len()):.3f}")

In [ ]:
## Cell H2 — PC1 prop 교란 검사 + gold 안 쓰는 프록시
kr = pd.read_parquet(P + "k_relabel_v2.parquet")
qd2 = qdf.merge(kr[["question","cb_greedy","cb_samples"]], on="question", how="left")

# ① PC1도 prop 교란인가 (PCA를 fold 안에서)
from sklearn.decomposition import PCA
def oof_pc(X, y, g, npc=1):
    o = np.zeros(len(y))
    for tr, te in GroupKFold(5).split(X, y, g):
        sc = StandardScaler().fit(X[tr]); p = PCA(npc).fit(sc.transform(X[tr]))
        o[te] = p.transform(sc.transform(X[te]))[:, 0]
    a = roc_auc_score(y, o)
    return (max(a, 1-a), o if a > .5 else -o)

a_all, pc_oof = oof_pc(Xv, yq, gq)
print(f"PC1 (fold 내부)  전체 {a_all:.3f}")
print("속성 고정:")
for p_ in qdf.prop.value_counts().head(6).index:
    i = (qdf.prop == p_).values
    if i.sum() >= 40 and len(np.unique(yq[i])) > 1:
        print(f"  {p_:15s} n={i.sum():3d}  {oof_pc(Xv[i], yq[i], gq[i])[0]:.3f}")

# prop 위 증분
_, o_ppc = oof(np.c_[Xp, pc_oof.reshape(-1,1)])
d = []
for _ in range(2000):
    i = np.concatenate([np.where(gq==u)[0] for u in rng.choice(uq, len(uq), True)])
    if len(np.unique(yq[i])) < 2: continue
    d.append(roc_auc_score(yq[i], o_ppc[i]) - roc_auc_score(yq[i], o_p[i]))
print(f"\nΔ(prop+PC1 − prop) = {np.mean(d):+.3f} CI [{np.percentile(d,2.5):+.3f}, {np.percentile(d,97.5):+.3f}]")

# ② gold 없는 프록시: 샘플 5개가 서로 일치하는 비율
import json as _j
def self_consist(s):
    v = _j.loads(s) if isinstance(s, str) else s
    return max(pd.Series(v).value_counts()) / len(v)
sc_ = qd2.cb_samples.map(self_consist)
print(f"\n자기일치도 (gold 미사용) AUROC={roc_auc_score(yq, sc_):.3f}")
print(f"출력 길이                AUROC={roc_auc_score(yq, qd2.cb_greedy.str.len()):.3f}")

In [ ]:
## Cell H3 — 자기일치도 검증 + 유사라벨 학습
# ① prop 교란 통과하나
_, o_sc = oof(np.c_[Xp, sc_.values.reshape(-1,1)])
d=[]
for _ in range(2000):
    i = np.concatenate([np.where(gq==u)[0] for u in rng.choice(uq,len(uq),True)])
    if len(np.unique(yq[i]))<2: continue
    d.append(roc_auc_score(yq[i],o_sc[i]) - roc_auc_score(yq[i],o_p[i]))
print(f"Δ(prop+자기일치도 − prop) = {np.mean(d):+.3f} CI [{np.percentile(d,2.5):+.3f}, {np.percentile(d,97.5):+.3f}]")

# ② 유사라벨로 학습한 probe가 진짜 K±를 맞히나  ← 핵심
y_pseudo = (sc_ >= 0.6).values.astype(int)
print(f"\n유사라벨 vs 실제 K± 일치율: {(y_pseudo==yq).mean():.3f}")
o = np.zeros(len(yq))
for tr, te in GroupKFold(5).split(Xv, yq, gq):
    p = make_pipeline(StandardScaler(), LogisticRegression(C=1, max_iter=3000))
    p.fit(Xv[tr], y_pseudo[tr])                    # 학습은 유사라벨
    o[te] = p.predict_proba(Xv[te])[:,1]
print(f"유사라벨 학습 → 실제 K± 평가: {roc_auc_score(yq, o):.3f}  (gold 학습 시 0.781)")

# ③ 속성 통제
_, o_pp = oof(np.c_[Xp, o.reshape(-1,1)])
d=[]
for _ in range(2000):
    i = np.concatenate([np.where(gq==u)[0] for u in rng.choice(uq,len(uq),True)])
    if len(np.unique(yq[i]))<2: continue
    d.append(roc_auc_score(yq[i],o_pp[i]) - roc_auc_score(yq[i],o_p[i]))
print(f"Δ(prop+유사라벨probe − prop) = {np.mean(d):+.3f} CI [{np.percentile(d,2.5):+.3f}, {np.percentile(d,97.5):+.3f}]")

In [ ]:
## Cell H4 — 임계값 민감도 + 연속 유사라벨
for th in [0.4, 0.6, 0.8, 1.0]:
    yp = (sc_ >= th).values.astype(int)
    if yp.mean() in (0, 1): continue
    o = np.zeros(len(yq))
    for tr, te in GroupKFold(5).split(Xv, yq, gq):
        p = make_pipeline(StandardScaler(), LogisticRegression(C=1, max_iter=3000))
        p.fit(Xv[tr], yp[tr]); o[te] = p.predict_proba(Xv[te])[:,1]
    print(f"th={th}  유사라벨 K+비율={yp.mean():.2f}  실제K± AUROC={roc_auc_score(yq,o):.3f}")

# 연속값 회귀
from sklearn.linear_model import Ridge
o = np.zeros(len(yq))
for tr, te in GroupKFold(5).split(Xv, yq, gq):
    p = make_pipeline(StandardScaler(), Ridge(alpha=100))
    p.fit(Xv[tr], sc_.values[tr]); o[te] = p.predict(Xv[te])
print(f"\n연속 회귀 → 실제K± AUROC={roc_auc_score(yq, o):.3f}")

### 2차 실행

In [ ]:
## Cell R — 분석 환경 복구 (GPU 불필요)
import os, json, warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score

P = "/content/drive/MyDrive/Grad_share/data_share/P9/hidden_v2/"
m  = pd.read_csv(P + "p8s_behavior_v2.csv")
kr = pd.read_parquet(P + "k_relabel_v2.parquet")
H  = {k: np.load(P + f"p8s_v2_{k}.npy", mmap_mode="r") for k in ("h_out","mlp_out")}

qdf = m.drop_duplicates("question").reset_index()
qdf = qdf.merge(kr[["question","cb_samples"]], on="question", how="left")
yq  = (qdf.kside_k == "K+").values.astype(int)
gq  = qdf.question.values
Xp  = pd.get_dummies(qdf.prop).values.astype(float)
Xv  = np.asarray(H["h_out"][qdf["index"].values, 13]).astype(np.float32)

def oof(X, y=yq, g=gq):
    p = make_pipeline(StandardScaler(), LogisticRegression(C=1, max_iter=3000))
    o = np.zeros(len(y))
    for tr, te in GroupKFold(5).split(X, y, g):
        p.fit(X[tr], y[tr]); o[te] = p.predict_proba(X[te])[:, 1]
    return roc_auc_score(y, o), o

def self_consist(s):
    v = json.loads(s) if isinstance(s, str) else s
    return max(pd.Series(v).value_counts()) / len(v)
sc_ = qdf.cb_samples.map(self_consist)

rng = np.random.RandomState(0); uq = np.unique(gq)
_, o_p = oof(Xp)

def delta(o_new, n=2000):
    d = []
    for _ in range(n):
        i = np.concatenate([np.where(gq==u)[0] for u in rng.choice(uq, len(uq), True)])
        if len(np.unique(yq[i])) < 2: continue
        d.append(roc_auc_score(yq[i], o_new[i]) - roc_auc_score(yq[i], o_p[i]))
    return np.mean(d), np.percentile(d, 2.5), np.percentile(d, 97.5)

print(f"n={len(qdf)}  prop 단독={roc_auc_score(yq, o_p):.3f}  "
      f"h_out L13={oof(Xv)[0]:.3f}  자기일치도={roc_auc_score(yq, sc_):.3f}")

In [ ]:
## Cell H6 — mlp_out 벡터
Xm = np.asarray(H["mlp_out"][qdf["index"].values, 13]).astype(np.float32)
a, o_m_ = oof(Xm)
_, o_pm = oof(np.c_[Xp, Xm])
mu, lo, hi = delta(o_pm)
print(f"mlp_out 벡터 L13  AUROC={a:.3f}   Δ={mu:+.3f} [{lo:+.3f}, {hi:+.3f}]")
print(f"  (h_out 벡터 0.781 / Δ +0.078,  mlp 노름 Δ +0.010,  PKS Δ +0.022)")

In [ ]:
## Cell H7 — attn_out도 되나 (FFN 특이성 확인)
Ha = np.load(P + "p8s_v2_attn_out.npy", mmap_mode="r")
Xa = np.asarray(Ha[qdf["index"].values, 13]).astype(np.float32)
a, _ = oof(Xa); _, o_pa = oof(np.c_[Xp, Xa])
mu, lo, hi = delta(o_pa)
print(f"attn_out 벡터 L13  AUROC={a:.3f}   Δ={mu:+.3f} [{lo:+.3f}, {hi:+.3f}]")

In [ ]:
## Cell H5 — diff-of-means 게이트 (training-free + gold-free)
y_ps = (sc_ >= 0.6).values.astype(int)

def dm_gate(X):
    o = np.zeros(len(yq))
    for tr, te in GroupKFold(5).split(X, yq, gq):
        s = StandardScaler().fit(X[tr]); Z1, Z2 = s.transform(X[tr]), s.transform(X[te])
        c = Z1[y_ps[tr]==1].mean(0) - Z1[y_ps[tr]==0].mean(0)
        o[te] = Z2 @ (c / np.linalg.norm(c))
    return o

for nm, X in [("h_out", Xv), ("mlp_out", Xm)]:
    o = dm_gate(X)
    _, o_pd = oof(np.c_[Xp, o.reshape(-1,1)])
    mu, lo, hi = delta(o_pd)
    print(f"{nm:8s} diff-of-means  AUROC={roc_auc_score(yq, o):.3f}  Δ={mu:+.3f} [{lo:+.3f}, {hi:+.3f}]")
print("  (logistic+유사라벨 0.772 / logistic+gold 0.781)")

### Steering 가능?

In [ ]:
!wget -q -O p8s_prompt.py https://raw.githubusercontent.com/Ontology0/Graduation-Project/refs/heads/dev/prompt/p8s_prompt.py
import hashlib, os
assert hashlib.md5(open("p8s_prompt.py","rb").read()).hexdigest() == "20959a4c34203ea0866d030baf23595d"
os.environ["HF_TOKEN"] = __import__("google.colab", fromlist=["userdata"]).userdata.get("HF_TOKEN")
print("ok")

In [ ]:
## Cell S1 — baseline 생성 (C− 612행)
import torch, time, json
from transformers import AutoTokenizer, AutoModelForCausalLM
from p8s_prompt import build_messages, truncate_answer, score_row, norm, MAX_NEW_TOKENS

MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL_ID, token=os.environ["HF_TOKEN"])
tok.padding_side = "left"
if tok.pad_token is None: tok.pad_token = tok.eos_token
EOS_IDS = {tok.eos_token_id, tok.convert_tokens_to_ids("<|eot_id|>")} - {None}
DECODE_KW = dict(skip_special_tokens=True, clean_up_tokenization_spaces=False)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, device_map="auto",
    attn_implementation="sdpa", token=os.environ["HF_TOKEN"]).eval()

cm = m[m.ctx_label == "C-"].reset_index(drop=True)          # 612행
cm["gold_list"] = cm.gold.apply(json.loads)
cm = cm.merge(kr[["question","cb_greedy"]], on="question", how="left")
prompts = [tok.apply_chat_template(build_messages(r.question, r.ctx_text),
           tokenize=False, add_generation_prompt=True) for r in cm.itertuples()]
order = np.argsort([len(tok(p, add_special_tokens=False)["input_ids"]) for p in prompts],
                   kind="stable")

@torch.no_grad()
def run(tag, hook_fn=None):
    out = [None]*len(prompts); handles = []
    if hook_fn is not None:
        for L in LAYERS: handles.append(model.model.layers[L].register_forward_hook(hook_fn(L)))
    t0 = time.time()
    for s in range(0, len(order), 16):
        idx = order[s:s+16]
        global CUR; CUR = idx                                  # 행별 coef 제어용
        enc = tok([prompts[i] for i in idx], return_tensors="pt",
                  padding=True, add_special_tokens=False).to(model.device)
        g = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                           pad_token_id=tok.pad_token_id, eos_token_id=list(EOS_IDS),
                           use_cache=True)[:, enc["input_ids"].shape[1]:]
        for j, i in enumerate(idx): out[i] = tok.decode(g[j], **DECODE_KW)
    for h in handles: h.remove()
    print(f"{tag} {time.time()-t0:.0f}s")
    return out

LAYERS = list(range(14, 19))
base = run("baseline")
cm["base_raw"] = base
cm["base_pred"] = [truncate_answer(x) for x in base]
sc = cm.apply(lambda r: score_row(r.base_pred, r.gold_list, r.answer_surface), axis=1)
cm[["base_em","base_sub","base_fc"]] = pd.DataFrame(sc.tolist(), index=cm.index)
# gold-free 행동 라벨
cm["b_retain"] = [int(norm(str(c)) in norm(p) and c==c) for p, c in zip(cm.base_pred, cm.cb_greedy)]
cm["b_accept"] = cm.base_fc
cm.to_parquet(P + "steer_baseline.parquet", index=False)
print(cm[["base_em","b_retain","b_accept"]].mean().round(3).to_dict())

In [ ]:
## Cell S2 — w 재산출 (OOF + prop 균형 + gold-free 라벨) & 게이트 점수
from sklearn.model_selection import GroupKFold

LAYERS = list(range(14, 19))          # 0-base
GATE_L = 13
ridx   = cm["index"].values if "index" in cm else \
         m.reset_index().merge(cm[["question","ctx_label"]], on=["question","ctx_label"])["index"].values
Hc     = {L: np.asarray(H["h_out"][ridx, L]).astype(np.float32) for L in LAYERS + [GATE_L]}
qc     = cm.question.values
props  = cm.prop.values

# 게이트 유사라벨 (자기일치도) — 질문 단위 값을 행에 붙임
sc_map = dict(zip(qdf.question, sc_))
gate_y = (cm.question.map(sc_map) >= 0.6).values.astype(int)

W      = {L: np.zeros((len(cm), 4096), np.float32) for L in LAYERS}   # 행별 OOF w
gate_s = np.zeros(len(cm), np.float32)
SIGMA  = {L: float(Hc[L].std()) for L in LAYERS}

def prop_balanced_diff(Z, pos, neg, pr):
    """속성별 diff-of-means를 구해 평균. prop 교란 제거."""
    ds = []
    for p_ in np.unique(pr):
        i = pr == p_
        if (pos & i).sum() >= 3 and (neg & i).sum() >= 3:
            ds.append(Z[pos & i].mean(0) - Z[neg & i].mean(0))
    if not ds:                                          # 폴백: 전체
        ds = [Z[pos].mean(0) - Z[neg].mean(0)]
    d = np.mean(ds, 0)
    return d / (np.linalg.norm(d) + 1e-8)

ret = (cm.b_retain == 1).values
acc = (cm.b_accept == 1).values
print(f"retain {ret.sum()}  accept {acc.sum()}  둘다 {(ret&acc).sum()}  other {(~ret&~acc).sum()}")
ret_pure = ret & ~acc                                   # 순도 우선 (findings)
acc_pure = acc & ~ret

for tr, te in GroupKFold(5).split(cm, groups=qc):
    for L in LAYERS:
        W[L][te] = prop_balanced_diff(Hc[L], ret_pure & np.isin(np.arange(len(cm)), tr),
                                      acc_pure & np.isin(np.arange(len(cm)), tr), props)
    # 게이트도 같은 fold에서
    Z = Hc[GATE_L]; s = StandardScaler().fit(Z[tr])
    c = s.transform(Z[tr])[gate_y[tr]==1].mean(0) - s.transform(Z[tr])[gate_y[tr]==0].mean(0)
    gate_s[te] = s.transform(Z[te]) @ (c / np.linalg.norm(c))

cm["gate"] = gate_s
y_true = (cm.kside_k == "K+").values.astype(int)
print(f"게이트 AUROC (C− 612행) = {roc_auc_score(y_true, gate_s):.3f}")
print("σ:", {L: round(SIGMA[L], 3) for L in LAYERS})
print("w 폴드간 cos:", round(float(np.dot(W[16][0], W[16][-1])), 3))
np.save(P + "steer_w.npy", np.stack([W[L] for L in LAYERS]))

In [ ]:
## Cell S3 — steered 생성 (coef 3개)
COEFS = [2, 4, 6]
Wm = {L: torch.from_numpy(W[L]).to(model.device, model.dtype) for L in LAYERS}
SG = {L: SIGMA[L] for L in LAYERS}
CUR, CUR_COEF = None, 0.0

def mk_hook(L):
    def hook(mod, args, out):
        if CUR_COEF == 0.0: return out
        h = out[0] if isinstance(out, tuple) else out
        d = (CUR_COEF * SG[L]) * Wm[L][CUR]          # (B, 4096)
        h[:, -1, :] = h[:, -1, :] + d                # 마지막 위치 (prefill 끝 + 매 decode 스텝)
        return (h,) + out[1:] if isinstance(out, tuple) else h
    return hook

res = {}
for cf in COEFS:
    CUR_COEF = float(cf)
    o = run(f"coef+{cf}", mk_hook)
    p_ = [truncate_answer(x) for x in o]
    s_ = [score_row(p, g, a) for p, g, a in zip(p_, cm.gold_list, cm.answer_surface)]
    res[cf] = pd.DataFrame(s_, columns=["em","sub","fc"])
    cm[f"em_{cf}"], cm[f"fc_{cf}"] = res[cf].em.values, res[cf].fc.values
    cm[f"raw_{cf}"] = o
    print(f"  coef+{cf}: em={res[cf].em.mean():.3f} (base {cm.base_em.mean():.3f})  "
          f"accept={res[cf].fc.mean():.3f} (base {cm.b_accept.mean():.3f})")
CUR_COEF = 0.0
cm.to_parquet(P + "steer_all.parquet", index=False)

In [ ]:
## Cell S4 — 게이트 합성 + 임계값 스윕
from scipy.stats import binomtest

base_em = cm.base_em.values
gate    = cm.gate.values
kpos    = (cm.kside_k == "K+").values

print("τ    coverage  gated_em   ΔEM    Δ건수   oracle_em  전체개입")
for cf in COEFS:
    st = cm[f"em_{cf}"].values
    print(f"\n--- coef +{cf} ---")
    all_em = st.mean()
    orc    = np.where(kpos, st, base_em).mean()                 # 진짜 K±를 안다면
    for q_ in [1.0, 0.8, 0.6, 0.5, 0.4, 0.2]:
        tau = np.quantile(gate, 1 - q_)
        sel = gate >= tau
        g_em = np.where(sel, st, base_em).mean()
        d    = (np.where(sel, st, base_em) - base_em)
        n01, n10 = int(((base_em==0)&(d>0)).sum()), int(((base_em==1)&(d<0)).sum())
        p = binomtest(n10, n01+n10, 0.5).pvalue if n01+n10 else 1.0
        print(f"{q_:.1f}   {sel.mean():.2f}    {g_em:.3f}   {g_em-base_em.mean():+.3f}  "
              f"{int(d.sum()*len(d)/len(d)):+3d}   +{n01}/-{n10}  p={p:.1e}")
    print(f"     전체개입 {all_em:.3f} ({all_em-base_em.mean():+.3f})   "
          f"oracle {orc:.3f} ({orc-base_em.mean():+.3f})")

In [ ]:
## Cell S5 — C+ 행에도 돌려서 실제 순이득 계산
cp = m[m.ctx_label == "C+"].reset_index(drop=True)
cp["gold_list"] = cp.gold.apply(json.loads)
ridx_p = m.reset_index().merge(cp[["question","ctx_label"]], on=["question","ctx_label"])["index"].values

prompts = [tok.apply_chat_template(build_messages(r.question, r.ctx_text),
           tokenize=False, add_generation_prompt=True) for r in cp.itertuples()]
order = np.argsort([len(tok(p, add_special_tokens=False)["input_ids"]) for p in prompts], kind="stable")

# C+ 행의 w는 C−에서 학습한 걸 그대로 (배포 시 C를 모르므로)
Wfull = {L: np.tile(W[L].mean(0), (len(cp), 1)) for L in LAYERS}
Wm = {L: torch.from_numpy(Wfull[L]).to(model.device, model.dtype) for L in LAYERS}

CUR_COEF = 0.0
o = run("C+ baseline")
cp["base_em"] = [score_row(truncate_answer(x), g, a)[0]
                 for x, g, a in zip(o, cp.gold_list, cp.answer_surface)]
for cf in COEFS:
    CUR_COEF = float(cf)
    o = run(f"C+ coef+{cf}", mk_hook)
    cp[f"em_{cf}"] = [score_row(truncate_answer(x), g, a)[0]
                      for x, g, a in zip(o, cp.gold_list, cp.answer_surface)]
    print(f"  C+ coef+{cf}: em={cp[f'em_{cf}'].mean():.3f} (base {cp.base_em.mean():.3f})")
CUR_COEF = 0.0
cp.to_parquet(P + "steer_cplus.parquet", index=False)

print("\n=== 전체 1224행 순이득 (C 구분 없이 전부 개입) ===")
for cf in COEFS:
    d = (cm[f"em_{cf}"].sum() - cm.base_em.sum()) + (cp[f"em_{cf}"].sum() - cp.base_em.sum())
    print(f"  coef+{cf}: C− {cm[f'em_{cf}'].sum()-cm.base_em.sum():+.0f}건  "
          f"C+ {cp[f'em_{cf}'].sum()-cp.base_em.sum():+.0f}건  →  순 {d:+.0f}건")

In [ ]:
## Cell S6
# ① C+ 셀별 — K+C+와 K−C+가 상쇄되고 있지 않은지
for cf in COEFS:
    d = cp[f"em_{cf}"].values - cp.base_em.values
    print(f"coef+{cf}: " + "  ".join(
        f"{c} {d[(cp.kside_k==c.split('C')[0]).values].mean():+.3f}" for c in ["K+", "K−"]))

# ② coef 더 올리면? (+6이 최고이고 단조 증가)
# ③ 질문 단위 cluster bootstrap CI
rng2 = np.random.RandomState(0)
allq = np.concatenate([cm.question.values, cp.question.values])
for cf in COEFS:
    d = np.concatenate([cm[f"em_{cf}"].values - cm.base_em.values,
                        cp[f"em_{cf}"].values - cp.base_em.values])
    uq2 = np.unique(allq); bs = []
    for _ in range(2000):
        i = np.concatenate([np.where(allq==u)[0] for u in rng2.choice(uq2, len(uq2), True)])
        bs.append(d[i].mean())
    print(f"coef+{cf}: ΔEM={d.mean():+.4f} [{np.percentile(bs,2.5):+.4f}, {np.percentile(bs,97.5):+.4f}]")

### Steering 재시도

In [ ]:
## Cell 0 — 환경·데이터·모델
!pip -q install -U transformers accelerate scikit-learn scipy
from google.colab import drive, userdata
drive.mount('/content/drive')
import os, re, json, time, hashlib
import numpy as np, pandas as pd, torch, transformers
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

!wget -q -O p8s_prompt.py https://raw.githubusercontent.com/Ontology0/Graduation-Project/refs/heads/dev/prompt/p8s_prompt.py
PROMPT_MD5 = hashlib.md5(open("p8s_prompt.py","rb").read()).hexdigest()
assert PROMPT_MD5 == "20959a4c34203ea0866d030baf23595d", PROMPT_MD5
from p8s_prompt import build_messages, norm, PROMPT_VERSION, MODEL_ID, MAX_NEW_TOKENS
from p8s_prompt import truncate_answer as trunc_v2          # 기존 채점기(비교용)

P9  = "/content/drive/MyDrive/Grad_share/data_share/P9/"
P   = P9 + "hidden_v2/"
OUT = P + "steer_v2/"; os.makedirs(OUT, exist_ok=True)

m    = pd.read_csv(P + "p8s_behavior_v2.csv")
orig = pd.read_csv(P9 + "p8s_behavior.csv")
kr   = pd.read_parquet(P + "k_relabel_v2.parquet")
H    = np.load(P + "p8s_v2_h_out.npy", mmap_mode="r")
meta = json.load(open(P + "p8s_v2_meta.json"))

# 행 정렬: H 행 == 원본 csv 행 == m 행
assert len(m) == len(orig) == H.shape[0] == 1224
assert (m.question.values == orig.question.values).all()
assert (m.ctx_label.values == orig.ctx_label.values).all()
m = m.drop(columns=["cb_greedy"], errors="ignore")
m["rid"] = np.arange(len(m))
m = m.merge(kr[["question","cb_greedy"]], on="question", how="left", validate="many_to_one")
assert (m.rid.values == np.arange(1224)).all() and (m.index.values == m.rid.values).all()
assert m.groupby("question").ctx_label.apply(lambda s: sorted(s) == ["C+","C-"]).all()
m["gold_list"] = m.gold.apply(json.loads)
print(meta["prompt_version"], "|", meta["layer_index"])
print("cb_greedy NaN", m.cb_greedy.isna().sum(),
      "| 빈값", (m.cb_greedy.fillna("").map(norm) == "").sum(),
      "| cb==surf(C-)", ((m.ctx_label=="C-") & (m.cb_greedy.fillna("").map(norm) == m.answer_surface.map(norm))).sum())

from transformers import AutoTokenizer, AutoModelForCausalLM
tok = AutoTokenizer.from_pretrained(MODEL_ID, token=os.environ["HF_TOKEN"])
tok.padding_side = "left"
if tok.pad_token is None: tok.pad_token = tok.eos_token
EOS_IDS = sorted({tok.eos_token_id, tok.convert_tokens_to_ids("<|eot_id|>")} - {None})
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, device_map="auto",
    attn_implementation="sdpa", token=os.environ["HF_TOKEN"]).eval()
print(transformers.__version__, torch.__version__, torch.cuda.get_device_name(0))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 134.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 156.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 75.3 MB/s eta 0:00:00
Mounted at /content/drive
p8s-v2 | 0-based; index i == model.model.layers[i]
cb_greedy NaN 0 | 빈값 0 | cb==surf(C-) 0


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

5.17.0 2.11.0+cu128 NVIDIA A100-SXM4-80GB


In [ ]:
## Cell 1 — 질문 단위 fold 고정 (한 번 만들고 파일로 고정)
from sklearn.model_selection import StratifiedKFold
SPLIT_F = OUT + "folds.json"
qdf = m.drop_duplicates("question")[["question","prop","kside_k"]].reset_index(drop=True)

if os.path.exists(SPLIT_F):
    fold_of = json.load(open(SPLIT_F)); print("기존 split 로드")
else:
    strata = (qdf.kside_k + "|" + qdf.prop)
    vc = strata.value_counts()
    strata = np.where(strata.map(vc) >= 5, strata, qdf.kside_k + "|rare")   # 희소 속성 묶기
    skf = StratifiedKFold(5, shuffle=True, random_state=20260919)
    fold_of = {}
    for k, (_, te) in enumerate(skf.split(qdf, strata)):
        for q in qdf.question.iloc[te]: fold_of[q] = int(k)
    json.dump(fold_of, open(SPLIT_F, "w"), ensure_ascii=False)

m["fold"] = m.question.map(fold_of)
assert m.fold.notna().all() and m.groupby("question").fold.nunique().max() == 1
print(pd.crosstab(m.fold, m.cell_k))
print(pd.crosstab(m[m.ctx_label=="C-"].fold, m[m.ctx_label=="C-"].prop).iloc[:, :8])

cell_k  K+C+  K+C-  K−C+  K−C-
fold                          
0         69    69    54    54
1         68    68    55    55
2         69    69    53    53
3         70    70    52    52
4         70    70    52    52
prop  author  capital  capital of  color  composer  country  director  father
fold                                                                         
0         15       15           3      1         7       11         9      10
1         16       14           3      1         7       11         9       9
2         15       13           4      1         7       10         9       9
3         15       14           2      1         8       10        10       9
4         15       15           3      1         7       10        11       9


/usr/local/lib/python3.13/dist-packages/sklearn/model_selection/_split.py:812: UserWarning: The least populated class in y has only 4 members, which is less than n_splits=5.
  warnings.warn(


In [ ]:
## Cell 2 — 채점기·훅·생성기 + 노이즈 바닥 (baseline 2회 + 영벡터 훅)
ABBR = r"(?<!\bDr)(?<!\bMr)(?<!\bMs)(?<!\bMrs)(?<!\bSt)(?<!\bJr)(?<!\bSr)(?<!\b[A-Z])"
def trunc(raw):
    s = str(raw).strip().split("\n")[0].strip()
    s = re.sub(r"^(answer|the answer is)\s*:?\s*", "", s, flags=re.I)
    if len(s.split()) > 8:                                   # 문장형일 때만 자른다
        s = re.split(ABBR + r"\.\s+(?=[A-Z])", s)[0]
    return re.sub(r"[.,;:\s]+$", "", s.strip().strip('"\'')).strip()
for a, b in [("Dr. Dre","Dr. Dre"), ("T.S. Eliot","T.S. Eliot"),
             ("Ronald D. Moore","Ronald D. Moore"), ("Answer: Paris.","Paris")]:
    assert trunc(a) == b, (a, trunc(a))

def score(raw, golds, surf, cb):
    p, s = norm(trunc(raw)), norm(surf)
    c = norm(cb) if isinstance(cb, str) else ""
    return dict(
        pred   = trunc(raw),
        em     = int(any(p == norm(g) for g in golds)),                # 주지표
        fc     = int(s != "" and s in p),                              # 기존 accept 정의(부분포함)
        acc_x  = int(s != "" and p == s),                              # w 라벨용 accept (완전일치)
        ret_x  = int(c != "" and c != s and p == c),                   # w 라벨용 retain (완전일치)
        em_v2  = int(any(norm(trunc_v2(raw)) == norm(g) for g in golds)))  # 기존 채점기

LAYERS = list(range(14, 19))
class Steer:
    """L14–18 출력의 마지막 위치에 coef·σ_L·v_L 가산. prefill 끝 + 매 decode 스텝."""
    def __init__(self):
        self.vec, self.coef, self.sig = None, 0.0, {}
        self.calls, self.dose = {}, {}
        self.h = [model.model.layers[L].register_forward_hook(self._mk(L)) for L in LAYERS]
    def _mk(self, L):
        def hook(mod, args, out):
            if self.vec is None: return out
            h = out[0] if isinstance(out, (tuple, list)) else out
            d = (self.coef * self.sig[L]) * self.vec[L]
            self.calls[L] = self.calls.get(L, 0) + 1
            if h.shape[1] > 1:                                  # prefill: 상대 주입량 기록
                r = d.float().norm() / h[:, -1].float().norm(dim=-1).clamp_min(1e-6)
                self.dose.setdefault(L, []).append(r.mean().item())
            h[:, -1, :] += d.to(h.dtype)
            return out
        return hook
    def set(self, vec=None, coef=0.0, sig=None):
        self.vec = None if vec is None else {
            L: torch.as_tensor(np.asarray(vec[L]), device=model.device, dtype=model.dtype) for L in LAYERS}
        self.coef, self.sig = float(coef), (sig or {})
        self.calls, self.dose = {}, {}

steer = Steer()
BS = 16
DECODE_KW = dict(skip_special_tokens=True, clean_up_tokenization_spaces=False)
m["prompt"] = [tok.apply_chat_template(build_messages(q, c), tokenize=False, add_generation_prompt=True)
               for q, c in zip(m.question, m.ctx_text)]
m["plen"] = [len(tok(p, add_special_tokens=False).input_ids) for p in m.prompt]
FOLD_ROWS = {k: m[m.fold == k].sort_values(["plen","rid"], kind="stable").rid.values for k in range(5)}

@torch.no_grad()
def gen_fold(k):
    out, rids = {}, FOLD_ROWS[k]
    for s in range(0, len(rids), BS):
        b = rids[s:s+BS]
        enc = tok(m.prompt.values[b].tolist(), return_tensors="pt", padding=True,
                  add_special_tokens=False).to(model.device)
        assert torch.all(enc.attention_mask[:, -1] == 1)
        g = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                           pad_token_id=tok.pad_token_id, eos_token_id=EOS_IDS)[:, enc.input_ids.shape[1]:]
        for r, row in zip(b, g.tolist()):
            stop = next((i for i, t in enumerate(row) if t in EOS_IDS), None)
            n = stop if stop is not None else len(row)
            out[int(r)] = (tok.decode(row[:n], **DECODE_KW), n, int(stop is None))
    return out

def run_cond(name, vec_by_fold=None, coef=0.0, sig_by_fold=None):
    f = OUT + f"gen_{name}.parquet"
    if os.path.exists(f):
        print(f"skip {name}"); return pd.read_parquet(f)
    rows, t0, dose = [], time.time(), {}
    for k in range(5):
        steer.set(None if vec_by_fold is None else vec_by_fold[k], coef,
                  None if sig_by_fold is None else sig_by_fold[k])
        o = gen_fold(k)
        if vec_by_fold is not None:
            assert all(steer.calls.get(L, 0) > 0 for L in LAYERS), "훅 미실행"
            dose[k] = {L: round(float(np.mean(v)), 4) for L, v in steer.dose.items()}
        rows += [dict(rid=r, fold=k, raw=raw, n_tok=n, hit_limit=hit) for r, (raw, n, hit) in o.items()]
    steer.set(None)
    d = pd.DataFrame(rows).sort_values("rid").reset_index(drop=True)
    assert (d.rid.values == np.arange(1224)).all()
    sc = pd.DataFrame([score(r, g, s, c) for r, g, s, c in
                       zip(d.raw, m.gold_list, m.answer_surface, m.cb_greedy)])
    d = pd.concat([d, sc], axis=1)
    d["cond"], d["coef"] = name, coef
    d["prompt_version"], d["prompt_md5"] = PROMPT_VERSION, PROMPT_MD5
    d["tf_version"], d["layers"] = transformers.__version__, "14-18(0-base)"
    d["hook_pos"], d["sigma_def"] = "last_pos_prefill+decode", "grand_trainfold"
    d["scorer"] = "trunc_abbrfix"
    d["dose_json"] = json.dumps(dose)
    d.to_parquet(f, index=False)
    print(f"{name} {time.time()-t0:.0f}s  dose(fold0)={dose.get(0)}")
    return d

A = run_cond("base_A")
B = run_cond("base_B")
zero = {k: {L: np.zeros(4096, np.float32) for L in LAYERS} for k in range(5)}
one  = {k: {L: 1.0 for L in LAYERS} for k in range(5)}
Z = run_cond("zero_vec", zero, 6.0, one)

for nm, d in [("base_B", B), ("zero_vec", Z)]:
    print(f"{nm:9s} vs base_A  문자열 불일치 {(d.raw != A.raw).sum()}  em 불일치 {(d.em != A.em).sum()}")
c = A.merge(m[["rid","ctx_label","kside_k"]], on="rid")
print(c.groupby("ctx_label")[["em","em_v2","fc","acc_x","ret_x","n_tok","hit_limit"]].mean().round(3))
print(c[c.ctx_label=="C-"].groupby("kside_k")[["em","fc","acc_x","ret_x"]].mean().round(3))

base_A 48s  dose(fold0)=None
base_B 46s  dose(fold0)=None
zero_vec 47s  dose(fold0)={14: 0.0, 15: 0.0, 16: 0.0, 17: 0.0, 18: 0.0}
base_B    vs base_A  문자열 불일치 0  em 불일치 0
zero_vec  vs base_A  문자열 불일치 0  em 불일치 0
              em  em_v2     fc  acc_x  ret_x  n_tok  hit_limit
ctx_label                                                     
C+         0.802  0.789  0.725  0.668  0.083  3.052        0.0
C-         0.260  0.257  0.516  0.475  0.237  3.260        0.0
            em     fc  acc_x  ret_x
kside_k                            
K+       0.419  0.408  0.384  0.396
K−       0.053  0.658  0.594  0.030


In [ ]:
## Cell 3 — fold별 offline w + σ (학습 fold의 C− 행만)
LAB = A[["rid","acc_x","ret_x"]].merge(
    m[["rid","fold","ctx_label","prop","kside_k","question"]], on="rid")
assert (LAB.rid.values == np.arange(1224)).all()
HL  = {L: np.asarray(H[:, L]).astype(np.float32) for L in LAYERS}      # 5 × (1224,4096)
isC = (LAB.ctx_label == "C-").values
PR  = LAB.prop.values

def prop_bal_diff(X, pos, neg, min_n=3):
    ds, used = [], []
    for p in np.unique(PR[pos | neg]):
        i = PR == p
        if (pos & i).sum() >= min_n and (neg & i).sum() >= min_n:
            ds.append(X[pos & i].mean(0) - X[neg & i].mean(0)); used.append(p)
    d = np.mean(ds, 0)
    return d / np.linalg.norm(d), used

def labels(k, sub=None):
    tr = (LAB.fold != k).values & isC
    if sub is not None: tr &= sub
    return tr & (LAB.ret_x == 1).values, tr & (LAB.acc_x == 1).values

W, SIG, INFO = {}, {}, []
for k in range(5):
    pos, neg = labels(k)
    trall = (LAB.fold != k).values                                    # σ: 학습 fold 전체 행
    W[k]   = {L: prop_bal_diff(HL[L], pos, neg)[0] for L in LAYERS}
    SIG[k] = {L: float(HL[L][trall].std()) for L in LAYERS}
    INFO.append(dict(fold=k, n_retain=int(pos.sum()), n_accept=int(neg.sum()),
                     n_prop=len(prop_bal_diff(HL[16], pos, neg)[1])))
print(pd.DataFrame(INFO).to_string(index=False))

# ① fold 간 방향 일치 (모든 쌍)
iu = np.triu_indices(5, 1)
for L in LAYERS:
    C = np.array([[W[a][L] @ W[b][L] for b in range(5)] for a in range(5)])[iu]
    print(f"L{L} fold간 cos  min {C.min():.3f}  mean {C.mean():.3f}")

# ② split-half: fold0 학습셋을 질문 무작위 절반 2개로 나눠 각각 w → cos (10회)
rng = np.random.RandomState(0)
qs0 = LAB.question[(LAB.fold != 0).values & isC].unique(); sh = []
for _ in range(10):
    h = LAB.question.isin(set(rng.choice(qs0, len(qs0)//2, replace=False))).values
    w1 = prop_bal_diff(HL[16], *labels(0, h))[0]
    w2 = prop_bal_diff(HL[16], *labels(0, ~h))[0]
    sh.append(w1 @ w2)
print(f"split-half cos L16 (절반 표본)  mean {np.mean(sh):.3f}  sd {np.std(sh):.3f}")

# ③ 진단: w가 K± 방향과 얼마나 겹치나 (gold 사용, 진단 전용)
for k in range(5):
    tr = (LAB.fold != k).values & isC
    kp, km = tr & (LAB.kside_k=="K+").values, tr & (LAB.kside_k=="K−").values
    kd = HL[16][kp].mean(0) - HL[16][km].mean(0); kd /= np.linalg.norm(kd)
    print(f"fold{k}  cos(w, K방향) L16 = {W[k][16] @ kd:+.3f}")

# ④ 랜덤 대조 방향 (fold 무관, seed 고정)
rng = np.random.RandomState(12345)
RAND = {}
for s in range(3):
    RAND[s] = {}
    for L in LAYERS:
        v = rng.randn(4096).astype(np.float32); RAND[s][L] = v / np.linalg.norm(v)
print("cos(rand, w) L16:", [round(float(RAND[s][16] @ W[0][16]), 3) for s in range(3)])
print("σ fold0:", {L: round(SIG[0][L], 4) for L in LAYERS})

np.savez(OUT + "w_offline.npz",
         **{f"w_f{k}_L{L}": W[k][L] for k in range(5) for L in LAYERS},
         **{f"sig_f{k}_L{L}": SIG[k][L] for k in range(5) for L in LAYERS})
json.dump(INFO, open(OUT + "w_offline_info.json", "w"))

 fold  n_retain  n_accept  n_prop
    0       116       234      11
    1       116       241      11
    2       119       223      13
    3       118       228      11
    4       111       238      10
L14 fold간 cos  min 0.927  mean 0.947
L15 fold간 cos  min 0.947  mean 0.966
L16 fold간 cos  min 0.945  mean 0.966
L17 fold간 cos  min 0.934  mean 0.959
L18 fold간 cos  min 0.918  mean 0.953
split-half cos L16 (절반 표본)  mean 0.747  sd 0.036
fold0  cos(w, K방향) L16 = +0.592
fold1  cos(w, K방향) L16 = +0.556
fold2  cos(w, K방향) L16 = +0.595
fold3  cos(w, K방향) L16 = +0.602
fold4  cos(w, K방향) L16 = +0.582
cos(rand, w) L16: [-0.006, -0.008, 0.005]
σ fold0: {14: 0.1273, 15: 0.136, 16: 0.1497, 17: 0.169, 18: 0.1906}


In [ ]:
## Cell 4 — steering 생성 (w coef 그리드 + 대조군 + 진단 arm)
# 진단용 방향 (gold K 사용 — 배포 불가, 해석 전용)
KD, WP = {}, {}
for k in range(5):
    tr = (LAB.fold != k).values & isC
    kp, km = tr & (LAB.kside_k == "K+").values, tr & (LAB.kside_k == "K−").values
    KD[k], WP[k] = {}, {}
    for L in LAYERS:
        kd = prop_bal_diff(HL[L], kp, km)[0]
        wp = W[k][L] - (W[k][L] @ kd) * kd
        KD[k][L], WP[k][L] = kd, wp / np.linalg.norm(wp)
print("fold0 L16  cos(w⊥K, K) =", round(float(WP[0][16] @ KD[0][16]), 4),
      "| cos(w⊥K, w) =", round(float(WP[0][16] @ W[0][16]), 3))

COEFS = [-6, -4, -2, 2, 4, 6, 8]
R = {"base": A}
for cf in COEFS:
    R[f"w{cf:+d}"] = run_cond(f"w{cf:+d}", W, cf, SIG)
for s in range(3):
    R[f"rand{s}+6"] = run_cond(f"rand{s}+6", {k: RAND[s] for k in range(5)}, 6, SIG)
R["kdir+6"]   = run_cond("kdir+6", KD, 6, SIG)
R["wperpK+6"] = run_cond("wperpK+6", WP, 6, SIG)
print("완료:", list(R))

fold0 L16  cos(w⊥K, K) = -0.0 | cos(w⊥K, w) = 0.577
w-6 49s  dose(fold0)={14: 0.0938, 15: 0.0926, 16: 0.0909, 17: 0.0914, 18: 0.0899}
w-4 50s  dose(fold0)={14: 0.0625, 15: 0.062, 16: 0.0613, 17: 0.0616, 18: 0.0609}
w-2 47s  dose(fold0)={14: 0.0313, 15: 0.0312, 16: 0.031, 17: 0.031, 18: 0.0308}
w+2 47s  dose(fold0)={14: 0.0313, 15: 0.0314, 16: 0.0315, 17: 0.0313, 18: 0.0314}
w+4 47s  dose(fold0)={14: 0.0625, 15: 0.0631, 16: 0.0636, 17: 0.0628, 18: 0.063}
w+6 50s  dose(fold0)={14: 0.0938, 15: 0.095, 16: 0.0962, 17: 0.0942, 18: 0.0946}
w+8 59s  dose(fold0)={14: 0.1251, 15: 0.1271, 16: 0.1291, 17: 0.1256, 18: 0.1255}
rand0+6 48s  dose(fold0)={14: 0.0938, 15: 0.0935, 16: 0.094, 17: 0.0939, 18: 0.0931}
rand1+6 47s  dose(fold0)={14: 0.0938, 15: 0.0937, 16: 0.0932, 17: 0.0925, 18: 0.0929}
rand2+6 47s  dose(fold0)={14: 0.0938, 15: 0.0935, 16: 0.0935, 17: 0.0939, 18: 0.0931}
kdir+6 48s  dose(fold0)={14: 0.0938, 15: 0.095, 16: 0.0954, 17: 0.0938, 18: 0.094}
wperpK+6 47s  dose(fold0)={14: 0.0938, 

In [ ]:
## Cell 5 — 분석 (판정 기준 순서대로)
from scipy.stats import binomtest
isCm = (m.ctx_label == "C-").values
b_em = A.em.values

def mcn(b, s):
    up, dn = int(((b==0)&(s==1)).sum()), int(((b==1)&(s==0)).sum())
    return up, dn, (binomtest(up, up+dn, 0.5).pvalue if up+dn else 1.0)

def ci(d, n=5000, seed=0):          # C−/C+ 각각 1행=1질문이라 행 bootstrap = 질문 cluster bootstrap
    r = np.random.RandomState(seed)
    return np.percentile(d[r.randint(0, len(d), (n, len(d)))].mean(1), [2.5, 97.5])

rows = []
for name, d in R.items():
    wc = d.pred.fillna("").str.split().str.len().values
    for c, msk in [("C-", isCm), ("C+", ~isCm)]:
        s, b = d.em.values[msk], b_em[msk]
        up, dn, p = mcn(b, s); lo, hi = ci(s - b)
        rows.append(dict(cond=name, C=c, em=s.mean(), dEM=(s-b).mean(), lo=lo, hi=hi,
                         up=up, dn=dn, p=p, accept=d.fc.values[msk].mean(),
                         retain_x=d.ret_x.values[msk].mean(), over6=(wc[msk] > 6).mean(),
                         hit=d.hit_limit.values[msk].mean()))
T = pd.DataFrame(rows)

pos = T[(T.C == "C-") & T.cond.isin(["w+2","w+4","w+6","w+8"])].sort_values("p")
adj = np.maximum.accumulate([min(1.0, (len(pos)-i) * p) for i, p in enumerate(pos.p)])
T.loc[pos.index, "p_holm"] = adj
T.to_csv(OUT + "summary.csv", index=False)

pd.set_option("display.width", 200)
for c in ["C-", "C+"]:
    print(f"\n===== {c} =====")
    print(T[T.C == c].drop(columns="C").round(4).to_string(index=False))

# 기준 3: w+6 vs 랜덤 (C−, paired)
print("\n[3] w+6 vs 랜덤 (C−)")
w6 = R["w+6"].em.values[isCm]
for s in range(3):
    up, dn, p = mcn(R[f"rand{s}+6"].em.values[isCm], w6)
    print(f"  vs rand{s}: w만 맞음 {up} / 랜덤만 맞음 {dn}  p={p:.2e}")

# C− 행동 전이 (base → w+6)
def beh(d): return np.select([d.ret_x == 1, d.acc_x == 1], ["retain", "accept"], "other")
print("\n[전이] C− base → w+6")
print(pd.crosstab(pd.Series(beh(A)[isCm], name="base"),
                  pd.Series(beh(R["w+6"])[isCm], name="w+6"), margins=True))

# fold별 일관성
print("\n[fold별] C− ΔEM")
for name in ["w+4", "w+6", "w+8"]:
    d = R[name]
    print(f"  {name}: " + "  ".join(
        f"f{k} {(d.em.values - b_em)[isCm & (m.fold == k).values].mean():+.3f}" for k in range(5)))

# K별 (보고용, 판정 기준 아님)
print("\n[K별] C− ΔEM")
for name in ["w+6", "kdir+6", "wperpK+6", "rand0+6"]:
    d = R[name]
    print(f"  {name:9s} " + "  ".join(
        f"{k} {(d.em.values - b_em)[isCm & (m.kside_k == k).values].mean():+.3f}" for k in ["K+", "K−"]))

print("\n[주입량] w+6 fold0 ‖Δh‖/‖h‖:", json.loads(R["w+6"].dose_json.iloc[0]).get("0"))


===== C- =====
    cond     em     dEM      lo      hi  up  dn      p  accept  retain_x  over6    hit  p_holm
    base 0.2598  0.0000  0.0000  0.0000   0   0 1.0000  0.5163    0.2369 0.0000 0.0000     NaN
     w-6 0.1225 -0.1373 -0.1667 -0.1095   3  87 0.0000  0.6536    0.1176 0.0016 0.0000     NaN
     w-4 0.1650 -0.0948 -0.1193 -0.0703   2  60 0.0000  0.6127    0.1569 0.0033 0.0000     NaN
     w-2 0.2124 -0.0474 -0.0654 -0.0310   1  30 0.0000  0.5621    0.2010 0.0000 0.0000     NaN
     w+2 0.3121  0.0523  0.0343  0.0703  33   1 0.0000  0.4706    0.2778 0.0000 0.0000     0.0
     w+4 0.3480  0.0882  0.0654  0.1111  56   2 0.0000  0.4199    0.3007 0.0033 0.0000     0.0
     w+6 0.3611  0.1013  0.0768  0.1258  64   2 0.0000  0.3807    0.3072 0.0049 0.0000     0.0
     w+8 0.3824  0.1225  0.0948  0.1503  80   5 0.0000  0.3252    0.3203 0.0082 0.0016     0.0
 rand0+6 0.2614  0.0016 -0.0065  0.0098   4   3 1.0000  0.5131    0.2386 0.0016 0.0000     NaN
 rand1+6 0.2680  0.0082  0.0016  0

In [ ]:
## Cell 6 — w+6 변화 행 확인 + 사람 검증 시트
d6 = R["w+6"]
x = pd.DataFrame({"question": m.question, "K": m.kside_k, "surf": m.answer_surface,
                  "cb": m.cb_greedy, "gold": m.gold_list.map(lambda g: g[:4]),
                  "base": A.pred, "w6": d6.pred, "beh0": beh(A), "beh6": beh(d6),
                  "em0": A.em, "em6": d6.em})[isCm]
t = x[(x.beh0 == "accept") & (x.beh6 == "other")]
print(f"accept→other {len(t)}건 중 gold 맞힘(em=1): {t.em6.sum()}")
print(t[t.em6 == 0][["K","surf","cb","w6","gold"]].to_string())

chg = x[(x.em0 != x.em6) | (x.beh0 != x.beh6)].copy()
chg["label"] = ""          # GOLD / CTX / NONE — E0와 같은 규칙
chg.to_csv(OUT + "audit_w6_changes.csv", index=False, encoding="utf-8-sig")
print("검증 대상", len(chg), "행")

accept→other 43건 중 gold 맞힘(em=1): 12
       K                                  surf                                   cb                                                                                                                           w6                                                                                            gold
618   K+                      Saint Peter Port                           Georgetown  Saint George's is not mentioned in the text, however, the text does mention that the village is 16 km from the capital city                          [Georgetown, Garden City of the Caribbean, Georgetown, British Guiana]
622   K+                            Evan Rosen                      Agatha Christie                                                                                                                         Evan  [Agatha Christie, Agatha Mary Clarissa Christie, Agatha Mary Clarissa Miller, Mary Westmacott]
625   K+                        Strom Thurmond  